# discopipe

> Cloud-init fragment for the discopipe bot: dedicated non-sudo user, venv install, claude CLI, agent CWD, hardened systemd unit

In [ ]:
#| default_exp discopipe

In [ ]:
#| hide
from nbdev.showdoc import *

Runs as the dedicated `discopipe` user (design Decision 6): the login
user has passwordless sudo, and the bot drives
`claude -p --dangerously-skip-permissions`. Secrets live in
`/etc/discopipe/env` (installed manually post-boot, never in user_data);
until it exists the unit stays inactive via `ConditionPathExists`.

## `discopipe_service`

In [ ]:
#| export
from boxrecipe.services import check_service, write_file_cmd

_UNIT = """[Unit]
Description=discopipe - Discord passthrough to a headless coding agent CLI
After=network-online.target
Wants=network-online.target
ConditionPathExists=/etc/discopipe/env
StartLimitIntervalSec=60
StartLimitBurst=3

[Service]
User=discopipe
WorkingDirectory=/home/discopipe/agent
EnvironmentFile=/etc/discopipe/env
Environment=PATH=/home/discopipe/.local/bin:/usr/local/bin:/usr/bin:/bin
ExecStart=/opt/discopipe/bin/discopipe
Restart=on-failure
RestartSec=5
NoNewPrivileges=yes
ProtectSystem=strict
ReadWritePaths=/home/discopipe
PrivateTmp=yes

[Install]
WantedBy=multi-user.target
"""

_CLAUDE_MD = """Replies are read on a phone via Discord. Keep every reply under 1800
characters. Be terse. For long output, write it to a file and reply with
the path and a 3-line summary.
"""

def discopipe_service(
)->dict:  # validated service dict for the discopipe bot (no domain, install+unit cmds)
    """Service dict for discopipe: outbound-only Discord bot, no Caddy block."""
    cmds = [
        "useradd --create-home --shell /usr/sbin/nologin discopipe",
        "python3 -m venv /opt/discopipe",
        "/opt/discopipe/bin/pip install git+https://github.com/doyu/discopipe.git",
        "sudo -H -u discopipe bash -c 'curl -fsSL https://claude.ai/install.sh | bash'",
        "install -d -m 700 -o root -g root /etc/discopipe",
        "install -d -o discopipe -g discopipe /home/discopipe/agent",
        write_file_cmd("/home/discopipe/agent/CLAUDE.md", _CLAUDE_MD,
                       owner="discopipe:discopipe"),
        write_file_cmd("/etc/systemd/system/discopipe.service", _UNIT),
        "systemctl daemon-reload",
        "systemctl enable discopipe",
    ]
    return check_service({"name": "discopipe", "domain": None, "port": None,
                          "public": False, "packages": ["python3-venv"], "cmds": cmds})

In [ ]:
svc = discopipe_service()
assert svc["name"] == "discopipe" and svc["domain"] is None and svc["public"] is False
assert "python3-venv" in svc["packages"]

joined = "\n".join(svc["cmds"])
# user exists before anything installs or writes into its home
assert "useradd --create-home --shell /usr/sbin/nologin discopipe" in joined
assert joined.index("useradd") < joined.index("claude.ai/install.sh")
# installs: bot venv (unpinned, Decision 5) + claude CLI as the service user
assert "python3 -m venv /opt/discopipe" in joined
assert "/opt/discopipe/bin/pip install git+https://github.com/doyu/discopipe.git" in joined
assert "sudo -H -u discopipe bash -c 'curl -fsSL https://claude.ai/install.sh | bash'" in joined
# secrets dir (root-only) and agent CWD
assert "install -d -m 700 -o root -g root /etc/discopipe" in joined
assert "install -d -o discopipe -g discopipe /home/discopipe/agent" in joined
# agent instructions landed
assert "/home/discopipe/agent/CLAUDE.md" in joined and "1800" in joined
# the unit: identity, wait-state, network ordering, hardening, restart policy
for token in ("/etc/systemd/system/discopipe.service",
              "User=discopipe",
              "WorkingDirectory=/home/discopipe/agent",
              "EnvironmentFile=/etc/discopipe/env",
              "ConditionPathExists=/etc/discopipe/env",
              "After=network-online.target", "Wants=network-online.target",
              "Environment=PATH=/home/discopipe/.local/bin:/usr/local/bin:/usr/bin:/bin",
              "ExecStart=/opt/discopipe/bin/discopipe",
              "Restart=on-failure", "RestartSec=5",
              "StartLimitIntervalSec=60", "StartLimitBurst=3",
              "NoNewPrivileges=yes", "ProtectSystem=strict",
              "ReadWritePaths=/home/discopipe", "PrivateTmp=yes",
              "WantedBy=multi-user.target"):
    assert token in joined, token
assert "systemctl daemon-reload" in joined
assert "systemctl enable discopipe" in joined
# never a secret value or env-var assignment in the fragment
for name in ("DISCORD_TOKEN", "DISCOPIPE_USER_ID", "DISCOPIPE_CHANNEL_ID", "ANTHROPIC_API_KEY"):
    assert name not in joined, name

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()